# Customer 360 Accelerator
### Notebook 03 — Create Data Agent

---

> **Purpose:** Create a Fabric Data Agent linked to the Semantic Model
> and Ontology, then publish it so it can answer natural-language queries
> about the Customer360 data.
>
> **Prerequisite:** Run notebooks `00`, `01`, and `02` first.

---
## Configuration

In [ ]:
# ── Paste outputs from previous notebooks, or leave blank to auto-detect ─────
WORKSPACE_ID      = ""
WORKSPACE_NAME    = ""
LAKEHOUSE_ID      = ""
LAKEHOUSE_NAME    = "Customer360Lakehouse"
TABLE_NAME        = "Customer360"
SEMANTIC_MODEL_ID = ""
ONTOLOGY_ID       = ""

DATAAGENT_NAME    = "Customer360Agent"

In [ ]:
import sempy.fabric as fabric
import time

client = fabric.FabricRestClient()

# Auto-detect workspace
if not WORKSPACE_ID:
    WORKSPACE_ID = fabric.get_notebook_workspace_id()
if not WORKSPACE_NAME:
    WORKSPACE_NAME = fabric.resolve_workspace_name()

# Auto-detect lakehouse
if not LAKEHOUSE_ID:
    resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/lakehouses")
    resp.raise_for_status()
    lh = next(
        (l for l in resp.json().get("value", []) if l["displayName"] == LAKEHOUSE_NAME),
        None,
    )
    if not lh:
        raise ValueError(f"Lakehouse '{LAKEHOUSE_NAME}' not found. Run Notebook 00 first.")
    LAKEHOUSE_ID = lh["id"]

# Auto-detect semantic model
if not SEMANTIC_MODEL_ID:
    resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/items?type=SemanticModel")
    resp.raise_for_status()
    sm = next(
        (i for i in resp.json().get("value", []) if i["displayName"] == LAKEHOUSE_NAME),
        None,
    )
    if sm:
        SEMANTIC_MODEL_ID = sm["id"]
    else:
        print("WARNING: Semantic Model not found. Agent will link to Lakehouse directly.")

# Auto-detect ontology
if not ONTOLOGY_ID:
    resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/ontologies")
    resp.raise_for_status()
    ont = next(
        (o for o in resp.json().get("value", []) if "Customer360" in o.get("displayName", "")),
        None,
    )
    if ont:
        ONTOLOGY_ID = ont["id"]
    else:
        print("WARNING: Ontology not found. Agent will be created without ontology.")

print(f"Workspace      : {WORKSPACE_NAME} ({WORKSPACE_ID})")
print(f"Lakehouse      : {LAKEHOUSE_NAME} ({LAKEHOUSE_ID})")
print(f"Semantic Model : {SEMANTIC_MODEL_ID or '(not found)'}")
print(f"Ontology       : {ONTOLOGY_ID or '(not found)'}")

---
## Step 1 — Check for Existing Data Agent

In [ ]:
DATAAGENT_ID = None

# Try dedicated dataAgents endpoint
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/dataAgents")
if resp.status_code == 200:
    for agent in resp.json().get("value", []):
        if agent.get("displayName") == DATAAGENT_NAME:
            DATAAGENT_ID = agent["id"]
            print(f"Data Agent already exists: {DATAAGENT_NAME} (ID: {DATAAGENT_ID})")
            break

# Fallback: check generic items API
if not DATAAGENT_ID:
    resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/items?type=DataAgent")
    if resp.status_code == 200:
        for item in resp.json().get("value", []):
            if item.get("displayName") == DATAAGENT_NAME:
                DATAAGENT_ID = item["id"]
                print(f"Data Agent found via Items API: {DATAAGENT_NAME} (ID: {DATAAGENT_ID})")
                break

if not DATAAGENT_ID:
    print(f"Data Agent '{DATAAGENT_NAME}' not found. Will create in next step.")

---
## Step 2 — Create Data Agent

Creates the Data Agent with the Semantic Model (or Lakehouse) as the
data source and the Ontology attached at creation time.

**Key design decision:** Data source and ontology are attached at
creation time (not patched post-hoc) — this follows Fabric's internal
provisioning order.

In [ ]:
import base64, json, uuid as _uuid

# -- Agent instructions -- live in stage_config.json (aiInstructions field) --
AGENT_INSTRUCTIONS = (
    "You are a Customer 360 analytics assistant for a business operating across India.\n\n"
    "Your data source is the Customer360 table which contains:\n"
    "- CustomerId: Unique customer identifier (C0001, C0002, ...)\n"
    "- FullName, State, City, Segment (Enterprise or Retail)\n"
    "- LifetimeValue, MonthlyRevenue (INR), ChurnRiskScore (0.0-1.0), LastPurchaseDate\n\n"
    "Guidance:\n"
    "- Flag ChurnRiskScore > 0.7 as high-priority for retention.\n"
    "- Aggregate MonthlyRevenue or LifetimeValue by Segment, State, or City.\n"
    "- Always include CustomerIds, counts, and numeric totals. Use INR for monetary values."
)


def _b64(obj):
    """Base64-encode a dict as compact JSON (UTF-8)."""
    return base64.b64encode(json.dumps(obj).encode()).decode()


def build_definition_parts(
    workspace_id,
    semantic_model_id="",
    semantic_model_name="",
    lakehouse_id="",
    lakehouse_name="",
    ontology_id="",
    ontology_name="",
    published=True,
):
    """
    Builds the definition.parts list for a Fabric Data Agent using the
    official REST API definition format (Files/Config/... path structure).

    File schema contracts per Fabric REST API docs:
      data_agent.json   -> {"$schema": "2.1.0"}
      datasource.json   -> {"$schema": "1.0.0", "type": ..., "artifactId": ..., ...}
      fewshots.json     -> {"$schema": "1.0.0", "fewShots": [...]}
      stage_config.json -> {"$schema": "1.0.0", "aiInstructions": "..."}
      publish_info.json -> {"$schema": "1.0.0", "description": "..."}
    """
    parts = []

    def _part(path, data):
        return {"path": path, "payload": _b64(data), "payloadType": "InlineBase64"}

    # -- data_agent.json: schema version marker ONLY --
    # Instructions do NOT go here -- they belong in stage_config.json
    parts.append(_part("Files/Config/data_agent.json", {"$schema": "2.1.0"}))

    # -- Semantic Model datasource (preferred) --
    if semantic_model_id:
        sm_name = semantic_model_name or "Customer360Lakehouse"
        ds_key = f"semantic_model-{sm_name}"
        ds = {
            "$schema": "1.0.0",
            "type": "semantic_model",
            "artifactId": semantic_model_id,
            "displayName": sm_name,
            "workspaceId": workspace_id,
        }
        fs = {"$schema": "1.0.0", "fewShots": []}
        parts.append(_part(f"Files/Config/draft/{ds_key}/datasource.json", ds))
        parts.append(_part(f"Files/Config/draft/{ds_key}/fewshots.json", fs))
        if published:
            parts.append(_part(f"Files/Config/published/{ds_key}/datasource.json", ds))
            parts.append(_part(f"Files/Config/published/{ds_key}/fewshots.json", fs))

    # -- Lakehouse datasource (fallback) --
    elif lakehouse_id:
        lh_name = lakehouse_name or "Customer360Lakehouse"
        ds_key = f"lakehouse-{lh_name}"
        ds = {
            "$schema": "1.0.0",
            "type": "lakehouse",
            "artifactId": lakehouse_id,
            "displayName": lh_name,
            "workspaceId": workspace_id,
        }
        fs = {"$schema": "1.0.0", "fewShots": []}
        parts.append(_part(f"Files/Config/draft/{ds_key}/datasource.json", ds))
        parts.append(_part(f"Files/Config/draft/{ds_key}/fewshots.json", fs))
        if published:
            parts.append(_part(f"Files/Config/published/{ds_key}/datasource.json", ds))
            parts.append(_part(f"Files/Config/published/{ds_key}/fewshots.json", fs))

    # -- Ontology (graph) datasource --
    # elements is REQUIRED: without it the GQL engine cannot resolve
    # label expressions like (node_Customer:`Customer`) and returns:
    #   "The label expression (Customer) does not match any node type"
    if ontology_id:
        ont_name = ontology_name or "Customer360Ontology"
        ds_key = f"graph-{ont_name}"
        ds = {
            "$schema": "1.0.0",
            "type": "graph",
            "artifactId": ontology_id,
            "displayName": ont_name,
            "workspaceId": workspace_id,
            "elements": [
                {
                    "id": str(_uuid.uuid4()),
                    "display_name": "Customer",
                    "type": "graph.nodeType",
                    "is_selected": True,
                }
            ],
        }
        fs = {"$schema": "1.0.0", "fewShots": []}
        parts.append(_part(f"Files/Config/draft/{ds_key}/datasource.json", ds))
        parts.append(_part(f"Files/Config/draft/{ds_key}/fewshots.json", fs))
        if published:
            parts.append(_part(f"Files/Config/published/{ds_key}/datasource.json", ds))
            parts.append(_part(f"Files/Config/published/{ds_key}/fewshots.json", fs))

    # -- stage_config.json: aiInstructions goes HERE (not in data_agent.json) --
    stage_cfg = {"$schema": "1.0.0", "aiInstructions": AGENT_INSTRUCTIONS}
    parts.append(_part("Files/Config/draft/stage_config.json", stage_cfg))

    # -- published/ mirror + publish_info.json --
    # Including publish_info.json marks the agent as Published at creation time,
    # removing the need for a separate /publish API call (returns 404 on many tenants).
    if published:
        parts.append(_part("Files/Config/published/stage_config.json", stage_cfg))
        parts.append(_part("Files/Config/publish_info.json", {
            "$schema": "1.0.0",
            "description": "Customer360Agent -- published at creation",
        }))

    return parts


if not DATAAGENT_ID:
    print(f"Creating Data Agent: {DATAAGENT_NAME}")

    definition_parts = build_definition_parts(
        workspace_id=WORKSPACE_ID,
        semantic_model_id=SEMANTIC_MODEL_ID,
        semantic_model_name=LAKEHOUSE_NAME,
        lakehouse_id=LAKEHOUSE_ID,
        lakehouse_name=LAKEHOUSE_NAME,
        ontology_id=ONTOLOGY_ID,
        ontology_name="Customer360Ontology",
        published=True,
    )

    print(f"   Definition parts: {len(definition_parts)}")

    create_payload = {
        "displayName": DATAAGENT_NAME,
        "description": "Customer360 conversational analytics agent",
        "definition": {"parts": definition_parts},
    }

    # Primary: dedicated dataAgents endpoint
    resp = client.post(
        f"v1/workspaces/{WORKSPACE_ID}/dataAgents",
        json=create_payload,
    )

    if resp.status_code in (200, 201):
        DATAAGENT_ID = resp.json().get("id")
        print(f"Data Agent created: {DATAAGENT_NAME} (ID: {DATAAGENT_ID})")

    elif resp.status_code == 202:
        op_id = resp.headers.get("x-ms-operation-id")
        retry_after = int(resp.headers.get("Retry-After", "15"))
        print(f"   Async creation started (op: {op_id}). Polling...")
        while True:
            time.sleep(retry_after)
            poll = client.get(f"v1/operations/{op_id}")
            poll.raise_for_status()
            status = poll.json().get("status")
            print(f"   Status: {status}")
            if status == "Succeeded":
                break
            elif status in ("Failed", "Cancelled"):
                print(f"   Agent creation {status}.")
                break
        list_resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/dataAgents")
        if list_resp.status_code == 200:
            for agent in list_resp.json().get("value", []):
                if agent.get("displayName") == DATAAGENT_NAME:
                    DATAAGENT_ID = agent["id"]
                    print(f"Data Agent ready: {DATAAGENT_NAME} (ID: {DATAAGENT_ID})")
                    break

    else:
        # Fallback: generic /items endpoint
        print(f"Primary endpoint failed (HTTP {resp.status_code}). Trying Items API fallback...")
        fallback_payload = {"type": "DataAgent", **create_payload}
        resp2 = client.post(
            f"v1/workspaces/{WORKSPACE_ID}/items",
            json=fallback_payload,
        )
        if resp2.status_code in (200, 201):
            DATAAGENT_ID = resp2.json().get("id")
            print(f"Data Agent created via Items API: {DATAAGENT_NAME} (ID: {DATAAGENT_ID})")
        elif resp2.status_code == 202:
            op_id = resp2.headers.get("x-ms-operation-id")
            if op_id:
                print(f"   Polling fallback operation {op_id}...")
                while True:
                    time.sleep(15)
                    poll = client.get(f"v1/operations/{op_id}")
                    poll.raise_for_status()
                    status = poll.json().get("status")
                    print(f"   Status: {status}")
                    if status in ("Succeeded", "Failed", "Cancelled"):
                        break
            list_resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/items?type=DataAgent")
            for item in list_resp.json().get("value", []):
                if item.get("displayName") == DATAAGENT_NAME:
                    DATAAGENT_ID = item["id"]
                    print(f"Data Agent ready: {DATAAGENT_NAME} (ID: {DATAAGENT_ID})")
                    break
        else:
            print(f"Fallback also failed (HTTP {resp2.status_code}): {resp2.text[:300]}")

    if DATAAGENT_ID:
        print("\nWaiting 20 seconds for Fabric to finish internal provisioning...")
        time.sleep(20)
else:
    print(f"Data Agent already exists -- skipping creation.")


---
## Step 3 — Publish the Data Agent

A newly created agent starts in **Draft** state. We need to publish it
to make it queryable. Multiple publish API endpoints are tried in sequence
since the available endpoint varies by Fabric tenant version.

In [ ]:
def is_agent_queryable(workspace_id, agent_id):
    """Check if the agent query endpoint is reachable (not 404)."""
    try:
        resp = client.post(
            f"v1/workspaces/{workspace_id}/dataAgents/{agent_id}/query",
            json={"userMessage": "ping"},
        )
        return resp.status_code != 404
    except Exception:
        return False


if DATAAGENT_ID:
    if is_agent_queryable(WORKSPACE_ID, DATAAGENT_ID):
        print(f"Agent '{DATAAGENT_NAME}' is already queryable -- no publish step needed.")
    else:
        print("Agent is in Draft state. Attempting to publish via updateDefinition...")

        # Primary publish method: updateDefinition with publish_info.json
        # This is the documented Fabric mechanism -- the presence of
        # publish_info.json + published/ parts transitions Draft -> Published.
        published_ok = False
        try:
            fresh_parts = build_definition_parts(
                workspace_id=WORKSPACE_ID,
                semantic_model_id=SEMANTIC_MODEL_ID,
                semantic_model_name=LAKEHOUSE_NAME,
                lakehouse_id=LAKEHOUSE_ID,
                lakehouse_name=LAKEHOUSE_NAME,
                ontology_id=ONTOLOGY_ID,
                ontology_name="Customer360Ontology",
                published=True,
            )
            upd_resp = client.post(
                f"v1/workspaces/{WORKSPACE_ID}/dataAgents/{DATAAGENT_ID}/updateDefinition",
                json={"definition": {"parts": fresh_parts}},
            )
            if upd_resp.status_code in (200, 201, 202, 204):
                print(f"   updateDefinition succeeded (HTTP {upd_resp.status_code})")
                published_ok = True
        except Exception as e:
            print(f"   updateDefinition failed (non-fatal): {e}")

        # Fallback: legacy publish endpoints
        if not published_ok:
            for endpoint in [
                f"v1/workspaces/{WORKSPACE_ID}/dataAgents/{DATAAGENT_ID}/publish",
                f"v1/workspaces/{WORKSPACE_ID}/dataAgents/{DATAAGENT_ID}/activate",
                f"v1/workspaces/{WORKSPACE_ID}/items/{DATAAGENT_ID}/publish",
            ]:
                try:
                    resp = client.post(endpoint, json={})
                    if resp.status_code in (200, 202, 204):
                        print(f"   Publish succeeded via {endpoint.split('/')[-1]}")
                        published_ok = True
                        break
                    else:
                        print(f"   {endpoint.split('/')[-1]}: HTTP {resp.status_code} -- trying next...")
                except Exception as ex:
                    print(f"   {endpoint.split('/')[-1]}: failed ({ex}) -- trying next...")

        if published_ok:
            print("\nWaiting 15 seconds for publish to take effect...")
            time.sleep(15)
            if is_agent_queryable(WORKSPACE_ID, DATAAGENT_ID):
                print(f"Agent '{DATAAGENT_NAME}' is now queryable!")
            else:
                print("Agent may still be activating. Try querying in a minute.")
        else:
            print(f"\n{'=' * 60}")
            print("  MANUAL PUBLISH REQUIRED")
            print(f"{'=' * 60}")
            print(f"  The programmatic publish API is not available on this tenant.")
            print(f"  Please publish manually:")
            print(f"  1. Go to: https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}")
            print(f"  2. Click '{DATAAGENT_NAME}'")
            print(f"  3. Click 'Publish' in the top ribbon")
            print(f"{'=' * 60}")
else:
    print("ERROR: No Data Agent ID available. Check previous steps.")


---
## Verification

In [ ]:
# List all Data Agents in workspace
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/dataAgents")
if resp.status_code == 200:
    agents = resp.json().get("value", [])
    print(f"Data Agents in workspace ({len(agents)} total):")
    for a in agents:
        marker = " <-- THIS ONE" if a["id"] == DATAAGENT_ID else ""
        print(f"   {a['displayName']} (ID: {a['id']}){marker}")
else:
    # Fallback
    resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/items?type=DataAgent")
    resp.raise_for_status()
    items = resp.json().get("value", [])
    print(f"Data Agent items ({len(items)} total):")
    for i in items:
        print(f"   {i['displayName']} (ID: {i['id']})")

In [ ]:
# Store IDs for downstream notebooks
print("\n" + "=" * 60)
print("OUTPUTS — Copy these to the next notebook")
print("=" * 60)
print(f"WORKSPACE_ID      = \"{WORKSPACE_ID}\"")
print(f"WORKSPACE_NAME    = \"{WORKSPACE_NAME}\"")
print(f"LAKEHOUSE_ID      = \"{LAKEHOUSE_ID}\"")
print(f"LAKEHOUSE_NAME    = \"{LAKEHOUSE_NAME}\"")
print(f"TABLE_NAME        = \"{TABLE_NAME}\"")
print(f"SEMANTIC_MODEL_ID = \"{SEMANTIC_MODEL_ID}\"")
print(f"ONTOLOGY_ID       = \"{ONTOLOGY_ID}\"")
print(f"DATAAGENT_ID      = \"{DATAAGENT_ID}\"")